By default, 40 example generations are recorded during the test run.

This is a simple notebook that allows you to view more generations (and reconstructions).

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

# model_path = "logs/nvae_acdc/latent-skip/pc-4-ws-6420-b-3/checkpoints/epoch=93-step=20116.ckpt"
# model_path = "logs/nvae_acdc/default/b1-11-b2-8-b3-9/checkpoints/epoch=99-step=21400.ckpt"

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
feats = get_data(loader_test)
feats.shape

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    feats = feats.to(device)
    
    num_data = feats.shape[0]
    samples_idx = torch.randperm(num_data)[:24]
    feats = feats[samples_idx]
    feats_hat_logits, _, _, _, _ = model(feats, test=True)

samples, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(feats, feats_hat_logits)
show_samples(samples, reconstruction_pixel_error, rgb=False, ncol=6, figsize=(24, 16), display=False)

In [ ]:
with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=24, device=device)
    feats_fake = model.conditional_coder(x_fake)

# Discretise probabilistic map then view generations
generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=6, figsize=(24, 16))